# Demand Planning Workflow

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eddmpython/vectrix/blob/master/notebooks/showcase/02_demand_planning_workflow.ipynb)

**A step-by-step demand planning workflow for operations teams.**

This notebook walks through a realistic monthly demand planning cycle:
1. Data quality check (anomalies, missing patterns)
2. Forecast generation with business constraints
3. Budget scenarios (optimistic / baseline / pessimistic)
4. Accuracy tracking vs. last month
5. Executive-ready Plotly reports

**Copy this notebook, replace the sample data with your CSV, and you have a working demand plan.**

In [ ]:
!pip install -q vectrix plotly

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from vectrix import forecast, analyze, loadSample
from vectrix import ConstraintAwareForecaster, Constraint
from vectrix.business import AnomalyDetector, WhatIfAnalyzer, BusinessMetrics

COLORS = {
    "primary": "#6366f1", "accent": "#a855f7",
    "positive": "#22c55e", "negative": "#ef4444", "warning": "#f59e0b",
    "muted": "#94a3b8", "bg": "#0f172a", "card": "#1e293b", "text": "#f1f5f9",
}
LAYOUT = dict(
    template="plotly_dark", paper_bgcolor=COLORS["bg"], plot_bgcolor=COLORS["bg"],
    font=dict(family="Inter, sans-serif", color=COLORS["text"]),
    margin=dict(l=60, r=30, t=60, b=40),
)

---

## 1. Load Your Data

Replace `loadSample("retail")` with `pd.read_csv("your_data.csv")` to use your own data.

In [ ]:
df = loadSample("retail")
DATE_COL = "date"
VALUE_COL = "sales"
HORIZON = 30

print(f"Rows: {len(df)} | Range: {df[DATE_COL].iloc[0]} ~ {df[DATE_COL].iloc[-1]}")
df.tail()

---

## 2. Data Quality Check

In [ ]:
report = analyze(df, date=DATE_COL, value=VALUE_COL)

detector = AnomalyDetector()
anomaly_result = detector.detect(df[VALUE_COL].values, method="auto")

print("=== Data Quality Summary ===")
print(f"  Observations:  {report.characteristics.length}")
print(f"  Period:        {report.characteristics.period}")
print(f"  Trend:         {report.characteristics.trendDirection} (str={report.characteristics.trendStrength:.2f})")
print(f"  Seasonality:   {'Yes' if report.characteristics.hasSeasonality else 'No'} (str={report.characteristics.seasonalStrength:.2f})")
print(f"  Anomalies:     {anomaly_result.nAnomalies} ({anomaly_result.anomalyRatio:.1%})")
print(f"  Changepoints:  {len(report.changepoints)}")
print(f"  Difficulty:    {report.dna.difficulty} ({report.dna.difficultyScore:.0f}/100)")

In [ ]:
dates = pd.to_datetime(df[DATE_COL])
values = df[VALUE_COL].values

fig = go.Figure()
fig.add_trace(go.Scatter(x=dates, y=values, name="Sales", line=dict(color=COLORS["muted"], width=1.5)))

if anomaly_result.nAnomalies > 0:
    fig.add_trace(go.Scatter(
        x=dates[anomaly_result.indices], y=values[anomaly_result.indices],
        name=f"Anomalies ({anomaly_result.nAnomalies})", mode="markers",
        marker=dict(color=COLORS["negative"], size=10, symbol="diamond"),
    ))

for cp in report.changepoints:
    fig.add_vline(x=dates[cp], line_dash="dash", line_color=COLORS["warning"], opacity=0.5)

fig.update_layout(**LAYOUT, title="Data Quality — Anomalies & Changepoints", height=400)
fig.show()

---

## 3. Generate Forecast with Business Constraints

In [ ]:
result = forecast(df, date=DATE_COL, value=VALUE_COL, steps=HORIZON)
print(f"Model: {result.model} | MAPE: {result.mape:.2f}%")

caf = ConstraintAwareForecaster()
constrained = caf.apply(
    result.predictions, result.lower, result.upper,
    constraints=[
        Constraint('non_negative', {}),
        Constraint('range', {'min': values.min() * 0.5, 'max': values.max() * 2.0}),
    ]
)

print(f"Constraints applied: non_negative + range [{values.min()*0.5:.0f}, {values.max()*2:.0f}]")

In [ ]:
forecast_df = result.toDataframe()
fc_dates = pd.to_datetime(forecast_df["date"])

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=dates[-60:], y=values[-60:],
    name="Recent History", line=dict(color=COLORS["muted"], width=1.5),
))

fig.add_trace(go.Scatter(x=fc_dates, y=result.upper, line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(
    x=fc_dates, y=result.lower, fill="tonexty", name="95% CI",
    fillcolor="rgba(99,102,241,0.12)", line=dict(width=0), hoverinfo="skip",
))

fig.add_trace(go.Scatter(
    x=fc_dates, y=constrained.predictions,
    name=f"Constrained Forecast ({result.model})",
    line=dict(color=COLORS["primary"], width=2.5),
))

fig.update_layout(**LAYOUT, title=f"Demand Forecast — Next {HORIZON} Days", height=450,
                  legend=dict(orientation="h", y=-0.15))
fig.show()

---

## 4. Budget Scenarios

In [ ]:
analyzer = WhatIfAnalyzer()
scenarios = analyzer.analyze(
    constrained.predictions,
    values,
    [
        {"name": "Baseline", "trend_change": 0.0},
        {"name": "Optimistic (+15%)", "trend_change": 0.15},
        {"name": "Pessimistic (-15%)", "trend_change": -0.15},
    ]
)

scenario_summary = pd.DataFrame([
    {
        "Scenario": s.name,
        "Total Volume": f"{s.predictions.sum():,.0f}",
        "Daily Avg": f"{s.predictions.mean():,.0f}",
        "Peak Day": f"{s.predictions.max():,.0f}",
        "Impact vs Base": f"{s.impact:+.1f}%",
    }
    for s in scenarios
])
scenario_summary

In [ ]:
scen_colors = [COLORS["primary"], COLORS["positive"], COLORS["negative"]]

fig = go.Figure()
for i, s in enumerate(scenarios):
    fig.add_trace(go.Scatter(
        x=fc_dates, y=s.predictions,
        name=s.name, line=dict(color=scen_colors[i], width=2, dash="solid" if i == 0 else "dash"),
    ))

fig.update_layout(**LAYOUT, title="Budget Scenarios — Optimistic / Baseline / Pessimistic",
                  height=400, legend=dict(orientation="h", y=-0.15))
fig.show()

---

## 5. Accuracy Tracking

Compare predicted vs. actual for the most recent period (simulated here).

In [ ]:
review_n = 30
actual_period = values[-review_n:]
np.random.seed(42)
predicted_period = actual_period * (1 + np.random.randn(review_n) * 0.04)

bm = BusinessMetrics()
metrics = bm.calculate(actual_period, predicted_period)

fig = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4],
                    subplot_titles=("Actual vs. Predicted", "Error Distribution"))

review_dates = dates[-review_n:]
fig.add_trace(go.Scatter(x=review_dates, y=actual_period, name="Actual",
                         line=dict(color=COLORS["text"], width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=review_dates, y=predicted_period, name="Predicted",
                         line=dict(color=COLORS["primary"], width=2, dash="dot")), row=1, col=1)

errors_pct = (predicted_period - actual_period) / actual_period * 100
fig.add_trace(go.Histogram(x=errors_pct, nbinsx=15, marker_color=COLORS["accent"],
                           name="Error %", showlegend=False), row=1, col=2)
fig.add_vline(x=0, line_dash="dash", line_color=COLORS["text"], row=1, col=2)

fig.update_layout(**LAYOUT, title=f"Accuracy Review — WAPE {metrics['wape']:.1f}%, Accuracy {metrics['forecastAccuracy']:.1f}%",
                  height=400, legend=dict(orientation="h", y=-0.15))
fig.show()

In [ ]:
print("=== Monthly Performance Review ===")
print(f"  Forecast Accuracy: {metrics['forecastAccuracy']:.1f}%")
print(f"  Bias:              {metrics['bias']:+.1f} units/day")
print(f"  Bias %:            {metrics['biasPercent']:+.2f}%")
print(f"  WAPE:              {metrics['wape']:.2f}%")
print(f"  MASE:              {metrics['mase']:.3f}")
print(f"  Over-forecast:     {metrics['overForecastRatio']:.0%}")
print(f"  Under-forecast:    {metrics['underForecastRatio']:.0%}")
print()
if metrics['mase'] < 1.0:
    print("  ✓ Model outperforms Naive baseline")
else:
    print("  ✗ Model underperforms Naive — investigate")
if abs(metrics['biasPercent']) > 5:
    direction = 'over' if metrics['bias'] > 0 else 'under'
    print(f"  ⚠ Systematic {direction}-forecasting detected")

---

## How to Use This for Your Business

1. **Replace the data source** — `pd.read_csv("your_sales.csv")` instead of `loadSample()`
2. **Adjust constraints** — set `min`, `max`, `capacity` to your business limits
3. **Define your scenarios** — growth rate, shock events relevant to your industry
4. **Schedule monthly** — run this notebook at the start of each planning cycle
5. **Share the Plotly charts** — they're interactive HTML, perfect for stakeholder reports

```python
# Your data
df = pd.read_csv("monthly_demand.csv")
result = forecast(df, date="order_date", value="units", steps=30)
```

---

**Resources:**
- [Vectrix Documentation](https://eddmpython.github.io/vectrix/docs/)
- [GitHub](https://github.com/eddmpython/vectrix)
- [PyPI](https://pypi.org/project/vectrix/)